<a href="https://colab.research.google.com/github/cc-huang-0716/internship_tests/blob/main/yolov8n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import shutil
import random
from google.colab import drive
import glob
from glob import glob

# 掛載 Drive
drive.mount('/content/drive')
anomaly_zip_path = "/content/drive/MyDrive/anomaly100.zip"
stable_zip_path = "/content/drive/MyDrive/stable500.zip"
extract_dir = "/content/dataset"

# 解壓縮
with zipfile.ZipFile(anomaly_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
with zipfile.ZipFile(stable_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# 設定資料夾名稱
#anomaly_dir = os.path.join(extract_dir, "anomaly_label")
stable_dir = os.path.join(extract_dir, "stable_series_images")

# 建立空白標註檔
for f in os.listdir(stable_dir):
    if f.endswith(".png") or f.endswith(".jpg"):
        txt_name = os.path.splitext(f)[0] + ".txt"
        txt_path = os.path.join(stable_dir, txt_name)
        if not os.path.exists(txt_path):
            open(txt_path, "w").close()

# 各自收集圖片路徑
#anomaly_images = [os.path.join(anomaly_dir, f) for f in os.listdir(anomaly_dir) if f.endswith(".jpg") or f.endswith(".png")]
stable_images  = [os.path.join(stable_dir, f) for f in os.listdir(stable_dir) if f.endswith(".jpg") or f.endswith(".png")]

base_dir = "/content/dataset"
stable_dir = os.path.join(base_dir, "stable_series_images")
merged_dir = os.path.join(base_dir, "merged")
os.makedirs(merged_dir, exist_ok=True)

# 取得空白背景圖與標註
stable_imgs = sorted([f for f in glob(f"{stable_dir}/*") if f.endswith(('.png', '.jpg'))])
random.shuffle(stable_imgs)
N = len(stable_imgs)
stable_train = stable_imgs[:int(N*0.8)]
stable_val   = stable_imgs[int(N*0.8):int(N*0.9)]
stable_test  = stable_imgs[int(N*0.9):]

splits = {
    "train": {"img": glob(f"{base_dir}/train/images/*"), "lbl": glob(f"{base_dir}/train/labels/*"), "stable": stable_train},
    "val": {"img": glob(f"{base_dir}/val/images/*"), "lbl": glob(f"{base_dir}/val/labels/*"), "stable": stable_val},
    "test": {"img": glob(f"{base_dir}/test/images/*"), "lbl": glob(f"{base_dir}/test/labels/*"), "stable": stable_test},
}

# 複製資料與標註
for split in splits:
    img_out = os.path.join(merged_dir, f"images/{split}")
    lbl_out = os.path.join(merged_dir, f"labels/{split}")
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    # 異常圖片與標註
    for img_path in splits[split]["img"]:
        fname = os.path.basename(img_path)
        shutil.copy(img_path, os.path.join(img_out, fname))
        label_path = os.path.join(base_dir, split, "labels", os.path.splitext(fname)[0] + ".txt")
        if os.path.exists(label_path):
            shutil.copy(label_path, os.path.join(lbl_out, os.path.basename(label_path)))

    # 正常圖片與空標註
    for img_path in splits[split]["stable"]:
        fname = os.path.basename(img_path)
        shutil.copy(img_path, os.path.join(img_out, fname))
        empty_txt = os.path.join(lbl_out, os.path.splitext(fname)[0] + ".txt")
        open(empty_txt, "w").close()

# 建立 data.yaml
with open(os.path.join(merged_dir, "data.yaml"), "w") as f:
    f.write(f"""
train: {merged_dir}/images/train
val: {merged_dir}/images/val
test: {merged_dir}/images/test

nc: 1
names: ['anomaly']
""")

print("已建立完成")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
已建立混合異常與正常圖的 YOLOv8 訓練資料結構！請使用 merged/data.yaml 開始訓練。


In [ ]:
import glob

for path in glob.glob(f"{extract_dir}/labels/*/*.txt"):
    with open(path, "r") as f:
        for i, line in enumerate(f):
            parts = line.strip().split()
            if len(parts) != 5:
                print(f"[格式錯誤] {path} line {i+1}: {line.strip()}")

In [ ]:
!pip install ultralytics
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data="/content/dataset/data.yaml", epochs=50, imgsz=640)
results = model.predict(
    source='/content/dataset/train/images',
    save=True,
    conf=0.5,
    imgsz=640,
    show=True
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 856.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 89.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

100%|██████████| 6.25M/6.25M [00:00<00:00, 44.5MB/s]


Ultralytics 8.3.169 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True,

100%|██████████| 755k/755k [00:00<00:00, 14.3MB/s]

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

...

      50/50         0G     0.5354     0.7675      0.843         12        640: 100%|██████████| 4/4 [00:53<00:00, 13.28s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:03<00:00,  3.70s/it]

                   all         20         20      0.997          1      0.995       0.83



50 epochs completed in 0.800 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train/weights/best.pt, 6.2MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.169 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:03<00:00,  3.39s/it]


                   all         20         20      0.997          1      0.995      0.865
Speed: 1.6ms preprocess, 155.6ms inference, 0.0ms loss, 2.5ms postprocess per image
Results saved to runs/detect/train
WARNING ⚠️ Environment does not support cv2.imshow() or PIL Image.show()


image 1/60 /content/dataset/train/images/series_001_png.rf.a124017aad3812974e50ab4d276ca7cf.jpg: 448x640 1 anomaly, 181.9ms
image 2/60 /content/dataset/train/images/series_002_png.rf.ae2b31c3e5ab6141409f5a3efa20888b.jpg: 448x640 1 anomaly, 170.6ms
image 3/60 /content/dataset/train/images/series_004_png.rf.fe4ef8a8a4aff79bdb2978b18345b1d8.jpg: 448x640 1 anomaly, 177.1ms
image 4/60 /content/dataset/train/images/series_005_png.rf.8c42a65ac27de15db039a30b8352fafd.jpg: 448x640 1 anomaly, 189.1ms
image 5/60 /content/dataset/train/images/series_006_png.rf.eda944192645409740ed77410b1dc271.jpg: 448x640 1 anomaly, 171.0ms
image 6/60 /content/dataset/train/images/series_009_png.rf.df827fcd3b12fcccfbf626768d1a113b.jpg: 

In [ ]:
model = YOLO("/content/runs/detect/train/weights/best.pt")
results = model.predict("/content/dataset/valid/images", save=True)

model = YOLO("/content/runs/detect/train/weights/best.pt")
model.train(
    data="dataset/data.yaml",
    epochs=50,
    imgsz=640
)


image 1/20 /content/dataset/valid/images/series_007_png.rf.62a86f7c545b580ea05f7ddddae3dc1b.jpg: 448x640 1 anomaly, 180.8ms
image 2/20 /content/dataset/valid/images/series_008_png.rf.c5486e6738a3abacc60c26c49647f440.jpg: 448x640 1 anomaly, 178.1ms
image 3/20 /content/dataset/valid/images/series_010_png.rf.0369a9c195e39d8d09e4d040d70dd986.jpg: 448x640 1 anomaly, 186.9ms
image 4/20 /content/dataset/valid/images/series_012_png.rf.9b099bf923237d203130b4a11d79acc4.jpg: 448x640 1 anomaly, 169.6ms
image 5/20 /content/dataset/valid/images/series_014_png.rf.98e9d9416a23186e89afc1da5a759a65.jpg: 448x640 1 anomaly, 165.2ms
image 6/20 /content/dataset/valid/images/series_033_png.rf.ad7b236a478b7c3e75c0c762cb81d13c.jpg: 448x640 1 anomaly, 172.1ms
image 7/20 /content/dataset/valid/images/series_040_png.rf.bbaaed3f7a9cb25532b3684f8e389335.jpg: 448x640 1 anomaly, 173.6ms
image 8/20 /content/dataset/valid/images/series_049_png.rf.b65015b1bd044c49cad453cd75cf9afa.jpg: 448x640 1 anomaly, 166.5ms
image 9

train: Scanning /content/dataset/train/labels.cache... 60 images, 0 backgrounds, 0 corrupt: 100%|██████████| 60/60 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1107.2±550.1 MB/s, size: 54.9 KB)



val: Scanning /content/dataset/valid/labels.cache... 20 images, 0 backgrounds, 0 corrupt: 100%|██████████| 20/20 [00:00<?, ?it/s]

Plotting labels to runs/detect/train3/labels.jpg... 


...

                   all         20         20      0.997          1      0.995      0.855



50 epochs completed in 0.798 hours.
Optimizer stripped from runs/detect/train3/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train3/weights/best.pt, 6.2MB

Validating runs/detect/train3/weights/best.pt...
Ultralytics 8.3.169 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:03<00:00,  3.32s/it]


                   all         20         20      0.997          1      0.995      0.869
Speed: 1.6ms preprocess, 154.3ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs/detect/train3


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c9a520e2b50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
results = model.predict("/content/dataset/merged/images/test", save=True)


image 1/70 /content/dataset/merged/images/test/series_003_png.rf.e53446c70bfaeeb854628569f2918022.jpg: 448x640 1 anomaly, 203.0ms
image 2/70 /content/dataset/merged/images/test/series_013_png.rf.518dfcaa90d5dd7dd4434b8edaa63ca9.jpg: 448x640 1 anomaly, 273.9ms
image 3/70 /content/dataset/merged/images/test/series_015_png.rf.cd4b8503e4e0fe6872427ffc8cd0be02.jpg: 448x640 1 anomaly, 211.0ms
image 4/70 /content/dataset/merged/images/test/series_018_png.rf.c2dd8fe5cb8539b17ce518ef2c9b2c3d.jpg: 448x640 1 anomaly, 223.8ms
image 5/70 /content/dataset/merged/images/test/series_024_png.rf.8e722b5cfc69db193f2dd968cf6a891e.jpg: 448x640 1 anomaly, 259.4ms
image 6/70 /content/dataset/merged/images/test/series_025_png.rf.54f19ff38a260d378e24a53213cea28b.jpg: 448x640 1 anomaly, 170.5ms
image 7/70 /content/dataset/merged/images/test/series_026_png.rf.809c51b0811f19a3e5fee554192971d2.jpg: 448x640 1 anomaly, 286.9ms
image 8/70 /content/dataset/merged/images/test/series_028_png.rf.7cb2a381ebb9271483dd0d9a

In [ ]:
metrics = model.val(split="test")
print(metrics.box.map50)
print(metrics.box.map)

Ultralytics 8.3.169 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 49.6±65.0 MB/s, size: 55.5 KB)


val: Scanning /content/dataset/test/labels... 20 images, 0 backgrounds, 0 corrupt: 100%|██████████| 20/20 [00:00<00:00, 695.83it/s]

val: New cache created: /content/dataset/test/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:04<00:00,  2.22s/it]


                   all         20         20      0.997          1      0.995       0.87
Speed: 4.3ms preprocess, 202.8ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to runs/detect/train33
0.995
0.8699285692679976
